# Whisper Medium LoRA 성능 비교

학습에 사용하지 않은 아동 화자의 test split에서 기본 Whisper Medium과 저장된 LoRA adapter를 비교합니다. CER를 주 지표로 사용하고 WER, 완전일치율, 평균 추론시간을 함께 CSV로 저장합니다.

In [ ]:
# Colab CUDA용 torch/torchaudio는 재설치하지 않습니다.
%pip install -q 'transformers>=4.46,<5' 'accelerate>=1,<2' 'peft>=0.13,<1' 'jiwer>=3,<5' 'librosa>=0.10,<1' 'soundfile>=0.12,<1' 'PyYAML>=6,<7' 'sentencepiece>=0.2,<1'

In [ ]:
import os
import subprocess
from pathlib import Path

import pandas as pd
import torch
import yaml
from google.colab import drive

assert torch.cuda.is_available(), 'Colab GPU 런타임을 선택해주세요.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
drive.mount('/content/drive')

REPO_URL = 'https://github.com/Mazu-Talk/MazuTalk.git'
BRANCH = 'feat/whisper-medium-lora-training'
REPO_DIR = Path('/content/MazuTalk')

if not (REPO_DIR / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)

In [ ]:
CONFIG_PATH = REPO_DIR / 'ai/stt/configs/whisper_medium_lora.yaml'
with CONFIG_PATH.open(encoding='utf-8') as file:
    config = yaml.safe_load(file)

archive_path = Path(config['data']['archive_path'])
data_root = Path(config['data']['runtime_root'])
child_metadata = data_root / config['data']['child_metadata']
asd_metadata = data_root / config['data']['asd_metadata']

if not child_metadata.exists() or not asd_metadata.exists():
    if not archive_path.exists():
        raise FileNotFoundError(f'Dataset archive not found: {archive_path}')
    data_root.mkdir(parents=True, exist_ok=True)
    subprocess.run(['tar', '-xf', str(archive_path), '-C', str(data_root)], check=True)

checkpoint_root = Path(config['training']['output_dir'])
print('data:', data_root)
print('checkpoint:', checkpoint_root)
print('child adapter:', (checkpoint_root / 'adapter-child').exists())
print('child+ASD adapter:', (checkpoint_root / 'adapter-child-asd').exists())

In [ ]:
# 300개면 비교 지표가 더 안정적입니다. 빠른 확인은 30~100으로 낮추세요.
EVAL_LIMIT = 300
SCRIPT_PATH = REPO_DIR / 'ai/stt/scripts/evaluate_whisper_lora.py'
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(
        f'{SCRIPT_PATH}가 없습니다. 로컬 변경사항을 commit/push한 뒤 clone 셀을 다시 실행해주세요.'
    )
print('evaluation script:', SCRIPT_PATH)
subprocess.run([
    'python',
    str(SCRIPT_PATH),
    '--config', str(CONFIG_PATH),
    '--child-limit', str(EVAL_LIMIT),
], check=True)

In [ ]:
output_dir = Path(config['evaluation']['output_dir'])
child_summary = pd.read_csv(output_dir / 'child_stt_metrics_summary.csv')
child_comparison = pd.read_csv(output_dir / 'child_stt_comparison.csv')
asd_summary = pd.read_csv(output_dir / 'asd_stt_metrics_summary.csv')

print('결과 저장 위치:', output_dir)
display(child_summary.style.format({
    'cer': '{:.4f}',
    'wer': '{:.4f}',
    'exact_match_accuracy': '{:.2%}',
    'cer_reduction_absolute': '{:.4f}',
    'cer_reduction_percent': '{:.2f}%',
}))
display(child_comparison.head(20))
display(asd_summary)

## 해석 기준

- `cer`가 낮을수록 좋습니다.
- `cer_reduction_percent`가 양수면 기본 Medium보다 개선된 것입니다.
- `exact_match_accuracy`가 높을수록 문장 전체를 정확히 인식한 비율이 높습니다.
- ASD 지표는 평가 발화가 2개뿐이므로 정량적 일반화 근거가 아니라 참고 결과로만 사용합니다.